# pycograd — part CPU, part GPU in one pass

A delegate backend (`torch`/`mps`) normally places **every** tensor of a differentiated pass on one device. `on_cpu(...)` / `on_device(...)` override that *per leaf*, so a single net — and a single autograd graph — can straddle CPU and GPU.

The classic use is an **embedding-table offload**: a large table is expensive to keep in GPU memory, so pin it on the CPU while the matmuls run on the GPU. The gather happens on CPU and only the small gathered slice crosses to the GPU; the table's gradient comes back on CPU and the big table never moves wholesale.

* You only tag the leaf: `table = on_cpu(big_table)`. The `$ |> ...` forward is **unchanged**.
* The backend **auto-unifies** devices where a CPU value meets a GPU value (torch errors on mixed-device ops otherwise), so `table[idx] @ w` "just works".
* `device` is a lifting hint: gradients still come back as numpy and the optimizer is unchanged. On the CPU `torch` fallback (no MPS) the tag is a harmless no-op, so this notebook runs anywhere.

In [1]:
%load_ext pycograd

## Setup

A tiny token-classification task: integer token ids → an embedding table → a linear classifier. The loss and activations are ordinary numpy used as pipe stages.

In [2]:
import logging
import numpy as np
from pycograd import params, frozen, on_cpu

def softmax_ce(logits, onehot):
    z = logits - np.max(logits, axis=1, keepdims=True)
    logp = z - np.log(np.sum(np.exp(z), axis=1, keepdims=True))
    return -np.mean(np.sum(onehot * logp, axis=1))

def relu(z): return np.maximum(0.0, z)

rng = np.random.default_rng(0)

# pick the GPU (MPS) backend, falling back to CPU torch when unavailable
import torch
logging.getLogger("torch._inductor.utils").setLevel(logging.ERROR)  # quiet a benign GPU perf note
BACKEND = "mps" if torch.backends.mps.is_available() else "torch"
print("compute device (backend):", repr(BACKEND), " — the embedding table is pinned to the CPU")

def train(weights, objective, steps, lr):
    first = last = None
    for _ in range(steps):
        value, grads = weights.grad(objective, backend=BACKEND, jit=True)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, lr)               # numpy SGD; each grad came back on its leaf's device
    return first, last

compute device (backend): 'mps'  — the embedding table is pinned to the CPU


## A CPU embedding table feeding a GPU classifier

`on_cpu[...]` pins the table to the CPU; the classifier `w`, `b` are untagged so they live on the compute device (the GPU). The forward gathers rows on the CPU and matmuls on the GPU — and the gradient from `weights.grad(..., backend="mps")` matches pycograd's numpy tape leaf-for-leaf (to float32 tolerance).

In [3]:
tok = np.array([0, 3, 7, 2, 5, 1, 9, 4])     # a batch of token ids
Yt  = np.eye(3)[[0, 1, 2, 0, 1, 2, 0, 1]]    # their labels (one-hot)

with params{
    table = on_cpu[0.1 * rng.standard_normal((16, 8))]   # 16 tokens x 8 dims — pinned to CPU
    w = 0.3 * rng.standard_normal((8, 3))                 # classifier — on the GPU
    b = np.zeros(3)
} as weights:
    forward = $ |> table[$] |> $ @ w + b                 # gather on CPU, matmul on GPU
    objective = lambda: forward(tok) |> softmax_ce($, Yt)
    v_np, g_np = weights.grad(objective)                         # numpy tape (reference)
    v_be, g_be = weights.grad(objective, backend=BACKEND, jit=True)
    first, last = train(weights, objective, 300, 0.5)

print("value   numpy=%.6f   %s=%.6f" % (float(v_np), BACKEND, float(v_be)))
print("grads match the numpy tape:",
      all(np.allclose(np.asarray(g_be[k]), np.asarray(g_np[k]), atol=1e-4, rtol=1e-3) for k in weights))
print("loss %.4f -> %.4f" % (first, last))

value   numpy=1.140446   mps=1.140446
grads match the numpy tape: True
loss 1.1404 -> 0.0020


## Where does each gradient live?

`weights.grad` always returns gradients as numpy, but *during* the pass each leaf is lifted to its home device, and torch's autograd returns each gradient on that device. We can watch the raw tensors cross the numpy boundary: the CPU table's gradient is computed on the **cpu**, the classifier's on the compute device — so the big table is never copied to the GPU.

In [4]:
import pycograd.backends.torch_backend as TB

seen = []                                    # (shape, device) of every tensor read back to numpy
orig = TB._torch_to_numpy
def spy(torch_mod, t):
    if isinstance(t, torch_mod.Tensor):
        seen.append((tuple(t.shape), t.device.type))
    return orig(torch_mod, t)
TB._torch_to_numpy = spy
try:
    with weights:
        forward = $ |> table[$] |> $ @ w + b
        weights.grad(lambda: forward(tok) |> softmax_ce($, Yt), backend=BACKEND)
finally:
    TB._torch_to_numpy = orig

dev = dict(seen)
print("table (16, 8) gradient computed on:", dev[(16, 8)])
print("classifier (8, 3) gradient computed on:", dev[(8, 3)])

table (16, 8) gradient computed on: cpu
classifier (8, 3) gradient computed on: mps


## Why bother: a table too big to want on the GPU

Make the table large — 100k tokens × 64 dims (~50 MB) — the kind you would not want resident in GPU memory. It stays on the CPU; each step gathers only the batch's rows and ships that small slice to the GPU. Training still runs entirely through `weights.grad(..., backend="mps", jit=True)`.

In [5]:
V, D, nclass = 100_000, 64, 4
batch = rng.integers(0, V, size=128)                 # 128 token ids out of 100k
Yb = np.eye(nclass)[rng.integers(0, nclass, size=128)]

with params{
    big = on_cpu[0.02 * rng.standard_normal((V, D))]  # ~50 MB embedding table — CPU-resident
    w = 0.1 * rng.standard_normal((D, nclass))        # classifier — on the GPU
    b = np.zeros(nclass)
} as weights:
    forward = $ |> big[$] |> relu |> $ @ w + b        # gather 128 rows on CPU, classify on GPU
    objective = lambda: forward(batch) |> softmax_ce($, Yb)
    first, last = train(weights, objective, 200, 0.2)

print("table shape %s (~%.0f MB) kept on CPU" % (weights["big"].value.shape, weights["big"].value.nbytes/1e6))
print("loss %.4f -> %.4f" % (first, last))

table shape (100000, 64) (~51 MB) kept on CPU
loss 1.3864 -> 1.2676


## A frozen, pretrained table

`on_cpu(...)` composes with `frozen(...)`: a pretrained embedding you don't want to fine-tune becomes a non-trainable, CPU-resident constant. Its gradient comes back `None`, the optimizer skips it, and only the classifier on the GPU learns.

In [6]:
pretrained = 0.1 * rng.standard_normal((16, 8))

with params{
    table = on_cpu[frozen[pretrained]]                # frozen + CPU-resident
    w = 0.3 * rng.standard_normal((8, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> table[$] |> $ @ w + b
    objective = lambda: forward(tok) |> softmax_ce($, Yt)
    _, g = weights.grad(objective, backend=BACKEND)
    first, last = train(weights, objective, 300, 0.5)

print("frozen table gradient:", g["table"])          # None -> excluded from autodiff
print("table left unchanged:", np.allclose(weights["table"].value, pretrained))
print("loss %.4f -> %.4f" % (first, last))

frozen table gradient: None
table left unchanged: True
loss 1.1494 -> 0.3856


## Notes & limits

* Tagging is per leaf: `on_cpu(x)` / `on_device("mps", x)` (and the `on_cpu[x]` subscript inside a `params{...}` block). It composes over `frozen`/`buffer`. Per-leaf placement needs the `torch`/`mps` backend (another backend rejects the tag with a clear error).
* Auto-unify always moves the off-device operand **to the compute device**, so structure nets so the large tensor is indexed/reduced *before* it would cross — exactly the embedding pattern here.
* See the GPU-compilation basics in [`pycograd_compile_mps_demo`](pycograd_compile_mps_demo.ipynb). The conformance suite:

```
python -m pytest test/test_device_placement.py
```